<a href="https://colab.research.google.com/github/Goseungeun/2026_BigData_Analyst/blob/main/Part4/Test_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1

In [72]:
# Q1
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch6/data6-1-1.csv')

df['출동시간'] = pd.to_datetime(df['출동시간'])
df['도착시간'] = pd.to_datetime(df['도착시간'])

df['diff'] = (df['도착시간'] - df['출동시간']).dt.total_seconds()/60

df = df.groupby('소방서')['diff'].mean()

print(round(df.max()))

81


In [74]:
# Q2
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch6/data6-1-2.csv')

df['spt'] = (df['1학년'] + df['2학년'] + df['3학년'] + df['4학년'] + df['5학년'] + df['6학년']) / df['교사수']
id_spt = df['spt'].idxmax()

print(df.iloc[id_spt]['교사수'])

19


In [82]:
# Q3
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch6/data6-1-3.csv')
print(df.head())

df['total_crime'] = df['강력범죄'] +df['절도범죄'] +df['폭력범죄'] +df['지능범죄'] +df['풍속범죄'] +df['교통범죄']
df['연도'] = df['날짜'].str[:4]

df = df.groupby('연도')['total_crime'].sum()
print(round(df.max()/12))

          날짜  강력범죄  절도범죄  폭력범죄  지능범죄  풍속범죄  교통범죄  경찰서명
0  2020년 01월    22   102    86    62    28   212  B경찰서
1  2020년 02월    26   138    80    61    31   183  E경찰서
2  2020년 03월    14   129    76    60    29   202  C경찰서
3  2020년 04월    26   142    83    71    33   182  B경찰서
4  2020년 05월    28   131    80    72    28   212  B경찰서
533


## Part 2

In [90]:
import pandas as pd
train = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch6/energy_train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch6/energy_test.csv')

print(train.shape)
print(test.shape)

print(train.info())
print(test.info())

target = train.pop('Heat_Load')

cols = train.columns[train.dtypes=='object'].tolist()
for col in cols:
  train_nu = train[col].nunique()
  test_nu = test[col].nunique()
  if train_nu != test_nu:
    print(col,'not same')
  else:
    print(col,'same num: ', train_nu)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

train = pd.get_dummies(train)
test = pd.get_dummies(test)

x_train,x_val,y_train,y_val = train_test_split(train,target,test_size = 0.2, random_state = 42)

rf = RandomForestClassifier()
rf.fit(x_train,y_train)
val_pred = rf.predict(x_val)
print(f1_score(y_val,val_pred,average='macro'))

pred = rf.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv('result.csv',index=False)

print(pd.read_csv('result.csv').shape)
print(pd.read_csv('result.csv').head())

(537, 10)
(231, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 537 entries, 0 to 536
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Compac       537 non-null    float64
 1   Surf_Area    537 non-null    float64
 2   Wall_Area    537 non-null    float64
 3   Roof         537 non-null    object 
 4   Height       537 non-null    object 
 5   Orient       537 non-null    object 
 6   Glaze_Area   537 non-null    float64
 7   Glaze_Distr  537 non-null    int64  
 8   Cool_Load    537 non-null    float64
 9   Heat_Load    537 non-null    object 
dtypes: float64(5), int64(1), object(4)
memory usage: 42.1+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Compac       231 non-null    float64
 1   Surf_Area    231 non-null    float64
 2   Wall_Area    231 non-null  

## Part 3

In [95]:
# Q1
import pandas as pd
df = pd.DataFrame({'항암약':[4,4,3,4,1,4,1,4,1,4,4,2,1,4,2,3,2,4,4,4]})

# 1-1
cnt = sum(df['항암약'] == 4)
print(cnt/len(df)) # 0.55

# 1-2 , 1-2
from scipy.stats import chisquare

prob = [0.1,0.05,0.15,0.7]
expected = [x*len(df) for x in prob]

observed = df.value_counts().sort_index().to_list()
print(chisquare(observed,expected)) # 검정통계량 : 6.976190476190476 , pvalue : 0.07266054733847573

0.55
Power_divergenceResult(statistic=np.float64(6.976190476190476), pvalue=np.float64(0.07266054733847573))


In [102]:
# Q2
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch6/data6-3-2.csv')
print(df.head())

# 2-1
from statsmodels.formula.api import ols
model = ols('temperature~solar+wind+o3',data=df).fit()
print(model.params['o3']) # 0.0749385437813658

# 2-2
print(model.pvalues['wind']) #0.7797177202071661

# 2-3
new = pd.DataFrame({'solar':[100],'wind':[5],'o3':[30]})
print(model.predict(new)) # 21.56163

    solar   wind     o3  temperature
0   89.14   6.28  33.52         23.0
1  109.97   1.04  27.01         20.7
2  102.83   6.42  41.00         20.5
3   84.94  10.20  33.44         22.2
4   94.21   4.95  29.97         21.4
0.0749385437813658
0.7797177202071661
0    21.56163
dtype: float64
